# 📐 Modelado de Datos — Persistencia en Hive
**Materia:** Herramientas de Software para Big Data  

**Modelo:** Normalizado  
**Fuente:** `/rfn/obligatorio/`  
**Destino:** Tablas Hive en esquema `inumet`

**Tablas del modelo:**
```
inumet.dim_estaciones     — datos de cada estacion meteorologica
inumet.dim_tiempo         — dimension temporal con atributos derivados
inumet.fact_temperatura   — mediciones de temperatura del aire
inumet.fact_viento        — mediciones de intensidad y direccion del viento
inumet.fact_precipitacion — mediciones de precipitacion horaria
inumet.fact_humedad       — mediciones de humedad relativa
inumet.fact_presion       — mediciones de presion atmosferica
inumet.fact_insolacion    — mediciones de horas de insolacion solar
```

## 1. Inicializacion de Spark con soporte Hive

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, month, dayofmonth, hour, when, trim
)
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("INUMET_Modelado_Hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo con soporte Hive")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-10T02:19:11,463 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Spark 3.4.1 listo con soporte Hive


## 2. Creacion del esquema en Hive

In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS inumet")
spark.sql("USE inumet")

print("Esquemas disponibles en Hive:")
spark.sql("SHOW DATABASES").show()

2026-06-10T02:24:43,458 INFO [Thread-4] org.apache.hadoop.hive.conf.HiveConf - Found configuration file file:/home/ort/spark/conf/hive-site.xml
2026-06-10T02:24:43,643 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.metastore.wm.default.pool.size does not exist
2026-06-10T02:24:43,643 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.task.scheduler.preempt.independent does not exist
2026-06-10T02:24:43,643 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.output.format.arrow does not exist
2026-06-10T02:24:43,643 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.tez.llap.min.reducer.per.executor does not exist
2026-06-10T02:24:43,643 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.arrow.root.allocator.limit does not exist
2026-06-10T02:24:43,643 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.vectorized.use.che

## 3. Carga desde /rfn

In [5]:
RFN = "hdfs://localhost:9000/rfn/obligatorio"

df_temp    = spark.read.parquet(f"{RFN}/temperatura")
df_viento  = spark.read.parquet(f"{RFN}/viento")
df_lluvia  = spark.read.parquet(f"{RFN}/precipitacion")
df_humedad = spark.read.parquet(f"{RFN}/humedad")
df_presion = spark.read.parquet(f"{RFN}/presion")
df_helio   = spark.read.parquet(f"{RFN}/heliofania")

print("Tablas cargadas desde /rfn:")
for nombre, df in [("temperatura", df_temp), ("viento", df_viento),
                   ("precipitacion", df_lluvia), ("humedad", df_humedad),
                   ("presion", df_presion), ("heliofania", df_helio)]:
    print(f"  {nombre:<15}: {df.count():>10,} registros")

Tablas cargadas desde /rfn:


[Stage 6:>                                                          (0 + 2) / 2]

  temperatura    :    371,841 registros


  viento         :    183,714 registros
  precipitacion  :    330,041 registros
  humedad        :    367,199 registros
  presion        :    365,937 registros
  heliofania     :      4,599 registros


## 4. dim_estaciones
Construida manualmente con los datos conocidos de cada estacion.

In [6]:
estaciones_data = [
    ("Aeropuerto Melilla G3", "Montevideo",  "costera",  -34.8335, -56.0303),
    ("Artigas G3",            "Artigas",     "interior", -30.4000, -56.5000),
    ("Colonia G3",            "Colonia",     "costera",  -34.4622, -57.8408),
    ("Mercedes G3",           "Soriano",     "interior", -33.2500, -58.0833),
    ("Paso de los Toros G3",  "Tacuarembo",  "interior", -32.8167, -56.5167),
    ("Rocha G3",              "Rocha",       "costera",  -34.4833, -54.3333),
    ("Salto G3",              "Salto",       "interior", -31.3833, -57.9667),
]

schema_est = StructType([
    StructField("estacion_id",  StringType(), False),
    StructField("departamento", StringType(), False),
    StructField("zona",         StringType(), False),
    StructField("latitud",      DoubleType(), True),
    StructField("longitud",     DoubleType(), True),
])

dim_estaciones = spark.createDataFrame(estaciones_data, schema=schema_est)
dim_estaciones.show(truncate=False)

[Stage 24:>                                                         (0 + 1) / 1]

+---------------------+------------+--------+--------+--------+
|estacion_id          |departamento|zona    |latitud |longitud|
+---------------------+------------+--------+--------+--------+
|Aeropuerto Melilla G3|Montevideo  |costera |-34.8335|-56.0303|
|Artigas G3           |Artigas     |interior|-30.4   |-56.5   |
|Colonia G3           |Colonia     |costera |-34.4622|-57.8408|
|Mercedes G3          |Soriano     |interior|-33.25  |-58.0833|
|Paso de los Toros G3 |Tacuarembo  |interior|-32.8167|-56.5167|
|Rocha G3             |Rocha       |costera |-34.4833|-54.3333|
|Salto G3             |Salto       |interior|-31.3833|-57.9667|
+---------------------+------------+--------+--------+--------+



## 5. dim_tiempo
Construida a partir de todas las fechas unicas de los datos.

In [7]:
fechas = df_temp.select("fecha") \
    .union(df_viento.select("fecha")) \
    .union(df_lluvia.select("fecha")) \
    .union(df_humedad.select("fecha")) \
    .union(df_presion.select("fecha")) \
    .union(df_helio.select("fecha")) \
    .distinct()

dim_tiempo = fechas \
    .withColumn("anio", year(col("fecha"))) \
    .withColumn("mes",  month(col("fecha"))) \
    .withColumn("dia",  dayofmonth(col("fecha"))) \
    .withColumn("hora", hour(col("fecha"))) \
    .withColumn("estacion_anio",
        when((col("mes") >= 12) | (col("mes") <= 2), "verano")
        .when((col("mes") >= 3)  & (col("mes") <= 5), "otonio")
        .when((col("mes") >= 6)  & (col("mes") <= 8), "invierno")
        .otherwise("primavera")
    ) \
    .withColumn("nombre_mes",
        when(col("mes") == 1,  "Enero").when(col("mes") == 2,  "Febrero")
        .when(col("mes") == 3,  "Marzo").when(col("mes") == 4,  "Abril")
        .when(col("mes") == 5,  "Mayo").when(col("mes") == 6,  "Junio")
        .when(col("mes") == 7,  "Julio").when(col("mes") == 8,  "Agosto")
        .when(col("mes") == 9,  "Septiembre").when(col("mes") == 10, "Octubre")
        .when(col("mes") == 11, "Noviembre").otherwise("Diciembre")
    ) \
    .orderBy("fecha")

print(f"dim_tiempo: {dim_tiempo.count():,} registros unicos")
dim_tiempo.show(5, truncate=False)

dim_tiempo: 56,423 registros unicos


+-------------------+----+---+---+----+-------------+----------+
|fecha              |anio|mes|dia|hora|estacion_anio|nombre_mes|
+-------------------+----+---+---+----+-------------+----------+
|2020-01-01 00:00:00|2020|1  |1  |0   |verano       |Enero     |
|2020-01-01 01:00:00|2020|1  |1  |1   |verano       |Enero     |
|2020-01-01 02:00:00|2020|1  |1  |2   |verano       |Enero     |
|2020-01-01 03:00:00|2020|1  |1  |3   |verano       |Enero     |
|2020-01-01 04:00:00|2020|1  |1  |4   |verano       |Enero     |
+-------------------+----+---+---+----+-------------+----------+
only showing top 5 rows



## 6. Tablas de hechos

In [8]:
fact_temperatura = df_temp.select(
    col("fecha"), col("estacion_id"), col("temp_aire")
)

fact_viento = df_viento.select(
    col("fecha"), col("estacion_id"), col("int_viento"), col("dir_viento")
)

fact_precipitacion = df_lluvia.select(
    col("fecha"), col("estacion_id"), col("precip_horario")
)

fact_humedad = df_humedad.select(
    col("fecha"), col("estacion_id"), col("hum_relativa")
)

fact_presion = df_presion.select(
    col("fecha"), col("estacion_id"), col("pres_atm_mar")
)

fact_insolacion = df_helio.select(
    col("fecha"), col("estacion_id"),
    col("heliofania").alias("horas_insolacion")
)

print("Tablas de hechos creadas:")
for nombre, df in [
    ("fact_temperatura",   fact_temperatura),
    ("fact_viento",        fact_viento),
    ("fact_precipitacion", fact_precipitacion),
    ("fact_humedad",       fact_humedad),
    ("fact_presion",       fact_presion),
    ("fact_insolacion",    fact_insolacion),
]:
    print(f"  {nombre:<25}: {df.count():>10,} registros | {df.columns}")

Tablas de hechos creadas:
  fact_temperatura         :    371,841 registros | ['fecha', 'estacion_id', 'temp_aire']
  fact_viento              :    183,714 registros | ['fecha', 'estacion_id', 'int_viento', 'dir_viento']
  fact_precipitacion       :    330,041 registros | ['fecha', 'estacion_id', 'precip_horario']
  fact_humedad             :    367,199 registros | ['fecha', 'estacion_id', 'hum_relativa']
  fact_presion             :    365,937 registros | ['fecha', 'estacion_id', 'pres_atm_mar']
  fact_insolacion          :      4,599 registros | ['fecha', 'estacion_id', 'horas_insolacion']


## 7. Persistencia en Hive
Se guardan todas las tablas en el esquema `inumet` de Hive.

In [9]:
tablas_hive = {
    "inumet.dim_estaciones":     dim_estaciones,
    "inumet.dim_tiempo":          dim_tiempo,
    "inumet.fact_temperatura":    fact_temperatura,
    "inumet.fact_viento":         fact_viento,
    "inumet.fact_precipitacion":  fact_precipitacion,
    "inumet.fact_humedad":        fact_humedad,
    "inumet.fact_presion":        fact_presion,
    "inumet.fact_insolacion":     fact_insolacion,
}

for nombre_tabla, df in tablas_hive.items():
    df.write.mode("overwrite").saveAsTable(nombre_tabla)
    print(f"  ✅ {nombre_tabla}")

print("\nTodas las tablas guardadas en Hive correctamente")

2026-06-10T02:26:33,093 INFO [Thread-4] org.apache.hadoop.hive.ql.security.authorization.plugin.sqlstd.SQLStdHiveAccessController - Created SQLStdHiveAccessController for session context : HiveAuthzSessionContext [sessionString=484c40d3-7945-429f-b2d5-546d5a2cb687, clientType=HIVECLI]
2026-06-10T02:26:33,096 WARN [Thread-4] org.apache.hadoop.hive.ql.session.SessionState - METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-06-10T02:26:33,097 INFO [Thread-4] hive.metastore - Mestastore configuration hive.metastore.filter.hook changed from org.apache.hadoop.hive.metastore.DefaultMetaStoreFilterHookImpl to org.apache.hadoop.hive.ql.security.authorization.plugin.AuthorizationMetaStoreFilterHook
2026-06-10T02:26:33,105 INFO [Thread-4] hive.metastore - Closed a connection to metastore, current connections: 0
2026-06-10T02:26:33,107 INFO [Thread-4] hive.metastore - Trying to connect to metastore with URI thrift://l

  ✅ inumet.dim_tiempo


  ✅ inumet.fact_temperatura


  ✅ inumet.fact_viento


  ✅ inumet.fact_precipitacion


  ✅ inumet.fact_humedad


  ✅ inumet.fact_presion
  ✅ inumet.fact_insolacion

Todas las tablas guardadas en Hive correctamente


## 8. Verificacion — tablas en Hive

In [10]:
print("Tablas en el esquema inumet:")
spark.sql("SHOW TABLES IN inumet").show(truncate=False)

print("\nConteo de registros por tabla:")
print(f"{'Tabla':<30} {'Registros':>12}")
print("-"*44)
for nombre_tabla in tablas_hive.keys():
    n = spark.sql(f"SELECT COUNT(*) as n FROM {nombre_tabla}").collect()[0]["n"]
    print(f"  {nombre_tabla:<28} {n:>12,}")

Tablas en el esquema inumet:
+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|inumet   |dim_estaciones    |false      |
|inumet   |dim_tiempo        |false      |
|inumet   |fact_humedad      |false      |
|inumet   |fact_insolacion   |false      |
|inumet   |fact_precipitacion|false      |
|inumet   |fact_presion      |false      |
|inumet   |fact_temperatura  |false      |
|inumet   |fact_viento       |false      |
+---------+------------------+-----------+


Conteo de registros por tabla:
Tabla                             Registros
--------------------------------------------
  inumet.dim_estaciones                   7
  inumet.dim_tiempo                  56,423
  inumet.fact_temperatura           371,841
  inumet.fact_viento                183,714
  inumet.fact_precipitacion         330,041
  inumet.fact_humedad               367,199
  inumet.fact_presion               365,937
  inumet.fact_insolacion 

## 9. Vista previa de las tablas en Hive
Consultamos las tablas directamente desde Hive para confirmar que funcionan.

In [11]:
print("=== inumet.dim_estaciones ===")
spark.sql("SELECT * FROM inumet.dim_estaciones").show(truncate=False)

print("\n=== inumet.dim_tiempo (muestra) ===")
spark.sql("SELECT * FROM inumet.dim_tiempo LIMIT 5").show(truncate=False)

print("\n=== inumet.fact_temperatura (muestra) ===")
spark.sql("SELECT * FROM inumet.fact_temperatura LIMIT 5").show(truncate=False)

print("\n=== inumet.fact_insolacion (muestra) ===")
spark.sql("SELECT * FROM inumet.fact_insolacion LIMIT 5").show(truncate=False)

=== inumet.dim_estaciones ===
+---------------------+------------+--------+--------+--------+
|estacion_id          |departamento|zona    |latitud |longitud|
+---------------------+------------+--------+--------+--------+
|Mercedes G3          |Soriano     |interior|-33.25  |-58.0833|
|Paso de los Toros G3 |Tacuarembo  |interior|-32.8167|-56.5167|
|Rocha G3             |Rocha       |costera |-34.4833|-54.3333|
|Salto G3             |Salto       |interior|-31.3833|-57.9667|
|Aeropuerto Melilla G3|Montevideo  |costera |-34.8335|-56.0303|
|Artigas G3           |Artigas     |interior|-30.4   |-56.5   |
|Colonia G3           |Colonia     |costera |-34.4622|-57.8408|
+---------------------+------------+--------+--------+--------+


=== inumet.dim_tiempo (muestra) ===
+-------------------+----+---+---+----+-------------+----------+
|fecha              |anio|mes|dia|hora|estacion_anio|nombre_mes|
+-------------------+----+---+---+----+-------------+----------+
|2020-01-01 00:00:00|2020|1  |1  

In [12]:
spark.stop()
print("Sesion Spark cerrada.")

Sesion Spark cerrada.
